# Amazon US Customer Reviews: Data Exploration

This notebook explores the data in the [Amazon US Customer Reviews](https://www.kaggle.com/datasets/cynthiarempel/amazon-us-customer-reviews-dataset) dataset sourced from Kaggle. The dataset is approximately 50.68 GB and contains over 109 million rows across dozens of product categories. We examine the data structure, distributions, missing values, duplicates, and key relationships between review attributes.

## Setup and Imports

In [1]:
from pyspark.sql import SparkSession
import requests
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SQLContext
from pyspark.sql.functions import col, length, round, expr, sum as spark_sum
from pyspark.ml.feature import Imputer

Matplotlib created a temporary cache directory at /scratch/ajaganathan/job_48870800/matplotlib-m6ypojt9 because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## Spark Session Configuration

Our SDSC Expanse allocation provides 16 total cores and 128 GB total memory. The dataset is 50.68 GB across 37 Parquet files. 

In [2]:
import subprocess
subprocess.run(["mkdir", "-p", "/home/ajaganathan/scratch/spark_tmp"])

CompletedProcess(args=['mkdir', '-p', '/home/ajaganathan/scratch/spark_tmp'], returncode=0)

In [3]:
spark = SparkSession.builder \
    .appName("AmazonReviewsCleaning") \
    .config("spark.driver.memory","8g") \
    .config("spark.executor.memory", "8g") \
    .config('spark.executor.instances', 15) \
    .config("spark.executor.cores", "2") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.local.dir", "/home/ajaganathan/scratch/spark_tmp") \
    .getOrCreate()

spark
print("local.dir:", spark.conf.get("spark.local.dir"))

local.dir: /home/ajaganathan/scratch/spark_tmp


### Executor Verification

In [4]:
# Get the active Spark Context, URL, and SQL Context
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"
sqlContext = SQLContext(spark.sparkContext)

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
spd = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
spd['maxMemory_GB'] = (spd['maxMemory'] / (1024**3)).round(2)
spd

/usr/local/spark/python/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,18,4965217075,0,True,4.62


### Current Spark Configuration

In [5]:
print("Executor Instances:", sc.getConf().get("spark.executor.instances"))
print("Executor Memory:", sc.getConf().get("spark.executor.memory"))
print("Driver Memory:", sc.getConf().get("spark.driver.memory"))
print("Executor Cores:", sc.getConf().get("spark.executor.cores"))
print("Total Cores Available:", sc._jsc.sc().defaultParallelism())

Executor Instances: 15
Executor Memory: 8g
Driver Memory: 8g
Executor Cores: 2
Total Cores Available: 18


## Data Loading

In [6]:
# Load the data into Spark dataframe
parquet_path = r"/expanse/lustre/projects/uci157/hkwon2/shared/amazon_reviews.parquet"
df = spark.read.parquet(parquet_path)

## Data Exploration

In [7]:
# Print the schema
df.printSchema()

root
 |-- marketplace: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- review_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_parent: integer (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- star_rating: string (nullable = true)
 |-- helpful_votes: integer (nullable = true)
 |-- total_votes: integer (nullable = true)
 |-- vine: string (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- review_headline: string (nullable = true)
 |-- review_body: string (nullable = true)
 |-- review_date: date (nullable = true)



In [ ]:
# Describe star rating, helpful votes, and total votes
df.describe("star_rating", "helpful_votes", "total_votes").show()

In [ ]:
# Row count and details on review_ids
total = df.count()
unique_reviews = df.select("review_id").distinct().count()
print(f"Total rows: {total}")
print(f"Unique review_ids: {unique_reviews}")
print(f"Duplicate review_ids: {total - unique_reviews}")

In [ ]:
# Missing values
df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df.columns
]).show()

In [ ]:
# Unique values
print("Unique categories:", df.select("product_category").distinct().count())
print("Unique marketplaces:", df.select("marketplace").distinct().count())

df.select("vine").distinct().show()
df.select("verified_purchase").distinct().show()

In [ ]:
# Star rating distribution
# Note: some rows contain dates, which will be filtered during preprocessing
df.groupBy("star_rating") \
  .count() \
  .orderBy("star_rating") \
  .show()

In [ ]:
# Reviews per Product Category 
# Note: some rows contain dates and reviews, which will be filtered during preprocessing
df.groupBy("product_category") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(200)

## Data Visualizations

### Star Rating Distrubtion

In [ ]:
star_data = df.filter(col("star_rating").isin(["1","2","3","4","5"])) \
    .groupBy("star_rating") \
    .count() \
    .orderBy("star_rating") \
    .toPandas()

plt.figure(figsize=(8,5))
plt.bar(star_data["star_rating"], star_data["count"], color="blue")
plt.title("Star Rating Distribution")
plt.xlabel("Star Rating")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

This bar chart shows the distribution of star ratings across all 109 million reviews. The distribution is heavily skewed toward 5-star ratings, which account for approximately 67 million reviews. 1-star reviews are the second most common at ~9.4 million, suggesting a J-shaped distribution where people are most motivated to review when they are either very satisfied or very disappointed. This class imbalance will need to be addressed during preprocessing.

### Top 10 Product Categories by Review Count

In [ ]:
cat_data = df.groupBy("product_category") \
    .count() \
    .orderBy("count", ascending=False) \
    .limit(10) \
    .toPandas()

plt.figure(figsize=(10,6))
plt.barh(cat_data["product_category"], cat_data["count"], color="orange")
plt.title("Top 10 Product Categories based on Review Count")
plt.xlabel("Number of Reviews")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

This horizontal bar chart displays the 10 most reviewed product categories. Wireless leads with approximately 9 million reviews, followed by PC and Mobile Apps. The distribution across categories is relatively uneven. Our analysis will need to account for this imbalance when comparing helpfulness across categories.

### Helpfulness Ratio: Verified vs Unverified Purchases

In [ ]:
verified_data = df.filter(
    (col("total_votes") > 0) &
    (col("verified_purchase").isin(["Y", "N"]))
).groupBy("verified_purchase") \
    .agg(round(
        spark_sum(col("helpful_votes")) / spark_sum(col("total_votes")), 3
    ).alias("helpfulness_ratio")) \
    .orderBy("verified_purchase") \
    .toPandas()

plt.figure(figsize=(6,5))
plt.bar(verified_data["verified_purchase"], verified_data["helpfulness_ratio"], color=["purple", "magenta"])
plt.title("Helpfulness Ratio: Verified vs Unverified Purchases")
plt.xlabel("Verified Purchase")
plt.ylabel("Helpfulness Ratio")
plt.tight_layout()
plt.show()

This bar chart compares the average helpfulness ratio between verified (Y = 0.75) and unverified (N = 0.72) purchases. Verified purchasers have a slightly higher helpfulness ratio, suggesting that shoppers marginally trust reviews from people who actually bought the product. The difference is small but consistent across 109M rows, making it statistically meaningful.

### Helpfulness Ratio by Star Rating

In [ ]:
star_help = df.filter(
    (col("total_votes") > 0) &
    (col("star_rating").isin(["1","2","3","4","5"]))
).groupBy("star_rating") \
  .agg(round(
      spark_sum(col("helpful_votes")) / spark_sum(col("total_votes")), 3
  ).alias("helpfulness_ratio")) \
  .orderBy("star_rating") \
  .toPandas()

plt.figure(figsize=(8,5))
plt.bar(star_help["star_rating"], star_help["helpfulness_ratio"], color="pink")
plt.title("Helpfulness Ratio by Star Rating")
plt.xlabel("Star Rating")
plt.ylabel("Helpfulness Ratio")
plt.tight_layout()
plt.show()

This bar chart shows a clear positive trend where higher star ratings match higher helpfulness ratios. 5 star reviews have a helpfulness ratio of ~0.82, while 1 star reviews have the lowest at ~0.56. This suggests that positive reviews are generally found more useful by other shoppers, which is an important insight.

### Review Length vs Helpfulness Ratio

In [ ]:
length_data = df.filter(
    (col("total_votes") > 0) &
    (col("review_body").isNotNull())
).withColumn("review_length", length(col("review_body"))) \
    .filter(col("review_length") < 5000) \
    .groupBy("review_length") \
    .agg(round(
        spark_sum(col("helpful_votes")) / spark_sum(col("total_votes")), 3
    ).alias("helpfulness_ratio")) \
    .orderBy("review_length") \
    .toPandas()

length_data = length_data.sample(frac=0.01, random_state=42)

plt.figure(figsize=(10,5))
plt.scatter(length_data["review_length"], length_data["helpfulness_ratio"], alpha=0.3, s=5, color="black")
plt.title("Review Length vs Helpfulness Ratio")
plt.xlabel("Review Length (by characters)")
plt.ylabel("Helpfulness Ratio")
plt.tight_layout()
plt.show()

This scatter plot shows the relationship between review length in characters and helpfulness ratio, sampled from reviews with at least 1 vote. A clear positive trend is visible which implies longer reviews tend to receive higher helpfulness ratios. Reviews under 500 characters cluster around 0.68 to 0.75, while reviews approaching 5,000 characters reach helpfulness ratios of 0.85 to 0.93. This supports our hypothesis that review length is a meaningful predictor of perceived helpfulness.

In [ ]:
# Create a temporary table/view "reviews" for the Spark dataframe
df.createOrReplaceTempView("reviews")

### Vine vs Non-Vine Helpfuless Ratio

In [ ]:
vine_pdf = sqlContext.sql(
    """
    SELECT vine, ROUND(AVG(helpful_votes / total_votes), 4) as avg_ratio
    FROM reviews
    WHERE total_votes > 0
    GROUP BY vine
    ORDER BY vine
    """).toPandas()

colors = ['green', 'blue']
bars = plt.bar(vine_pdf['vine'], vine_pdf['avg_ratio'], color=colors, width=0.6)
plt.bar_label(bars, padding=3, fmt='%.4f')
plt.title('Average Helpfulness Ratio for Vine vs Non-Vine')
plt.xlabel('Vine Member (N = No, Y = Yes)')
plt.ylabel('Average Helpfulness Ratio')
plt.ylim(0, 1)
plt.show()

This chart compares the average helpfulness ratio between Vine (Y=0.6412) and non-Vine (N=0.6761) reviewers. Amazon Vine is an invitation-only program where reviewers receive free products in exchange for reviews. Non-Vine reviewers have a slightly higher average helpfulness ratio. It's possible that shoppers trust reviews from people who paid for the product more than those who received it for free. 

### Average Helpfulness Ratio Per Year

In [ ]:
ratio_time_pdf = sqlContext.sql(
    """
    SELECT YEAR(review_date) as year, ROUND(AVG(helpful_votes / total_votes), 4) as avg_ratio
    FROM reviews
    WHERE total_votes > 0 AND YEAR(review_date) BETWEEN 1995 AND 2015
    GROUP BY year
    ORDER BY year
    """).toPandas()

plt.plot(ratio_time_pdf['year'], ratio_time_pdf['avg_ratio'], marker='o')
plt.title('Average Helpfulness Ratio Per Year')
plt.xlabel('Year')
plt.ylabel('Average Helpfulness Ratio')
plt.xticks(ratio_time_pdf['year'][::2].astype(int), rotation=45)
plt.ticklabel_format(style='plain', axis='y')
plt.show()

This line chart tracks the average helpfulness ratio of reviews from 1995 to 2015. The ratio declines as it gets closer to 2015.

### Number of Reviews Per Year

In [ ]:
time_pdf = sqlContext.sql(
    """
    SELECT YEAR(review_date) as year, COUNT(*) as count
    FROM reviews
    WHERE YEAR(review_date) BETWEEN 1950 AND 2026
    GROUP BY year
    ORDER BY year
    """).toPandas()

plt.plot(time_pdf['year'], time_pdf['count'], marker='o')
plt.xticks(time_pdf['year'][::2].astype(int), rotation=45)
plt.ticklabel_format(style='plain', axis='y')
plt.title('Number of Reviews Per Year')
plt.xlabel('Year')
plt.ylabel('Number of Reviews')
plt.show()

The line chart shows review volume over time. Growth is modest through 2010. It increases from just under 5 million reivews per year before 2011 to approximately 30 million reivews by 2015. 

Combined insight on Average Helpfulness Ratio Per Year and Number of Reviews Per Year:

The plot shows a decline in the average helpfulness ratio of Amazon reviews over time. While earlier reviews exhibit higher helpfulness, the ratio declines even as the review volume increases. The higher helpfulness scores could be due to a smaller and more engaged user base. The rapid growth may have introduced more low-effort, short, or spam reviews, which tend to receive fewer helpful votes.

## Preprocessing

### Clean below invalid data
1. Product category has invalid data like dates and reviews.
2. Star ratings has Null and dates
3. Null records in "review_id", "review_body", "review_headline", "review_date" and "product_category"
4. Duplicate review ids. Reviews have to be unique for analysis

In [ ]:
print(f"Before filtering invalid records: {df.count()}")

In [ ]:
#Valid categories of products
valid_categories = [
    "Wireless", "PC", "Mobile_Apps", "Digital_Ebook_Pur...", "Video DVD",
    "Apparel", "Music", "Health & Personal...", "Beauty", "Digital_Video_Dow...",
    "Toys", "Sports", "Shoes", "Books", "Automotive", "Electronics",
    "Office Products", "Pet Products", "Grocery", "Outdoors", "Camera",
    "Video Games", "Digital_Music_Pur...", "Baby", "Tools", "Watches",
    "Musical Instruments", "Furniture", "Video", "Software", "Gift Card",
    "Digital_Video_Games", "Mobile_Electronics", "Digital_Software",
    "Major Appliances", "Personal_Care_App...", "Home Entertainment",
    "Home Improvement", "Home", "Kitchen", "Lawn and Garden", "Luggage"
]

#Valid star ratings
valid_ratings = [1, 2, 3, 4, 5]

#Filter invalid product categories, star ratings, and votes
df_filtered = df.filter( \
         (col("product_category").isin(valid_categories)) & 
         (col("star_rating").isin(valid_ratings))) \
         .withColumn("helpful_votes", col("helpful_votes").cast("int")) \
         .withColumn("total_votes", col("total_votes").cast("int"))

In [ ]:
# Check Product Categories after removing invalid data
df_filtered.groupBy("product_category") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(50)

Invalid product categories have been filtered out

In [ ]:
# Check Star ratings after removing invalid data
df_filtered.groupBy("star_rating") \
  .count() \
  .orderBy("star_rating") \
  .show(50)

Invalid star ratings have been excluded

In [ ]:
#drop null values for dimensions that can't be imputed
valid_columns = ["review_id", "review_body", "review_headline", "review_date", "product_category"]

df_filtered = df_filtered.dropna(subset = valid_columns)

In [ ]:
#Drop duplicates
df_filtered = df_filtered.dropDuplicates(["review_id"])

In [ ]:
#impute median values for missing ratings and votes fields
df_clean = df_filtered.checkpoint()

#Impute median value for the null values
imputer = Imputer(
    strategy="median",
    inputCols=["star_rating", "helpful_votes", "total_votes"],
    outputCols=["star_rating", "helpful_votes", "total_votes"]
)

df_filtered = imputer.fit(df_filtered).transform(df_filtered)


In [ ]:
#Write the cleaned data to disk.
output_path = "/home/ajaganathan/scratch/spark_tmp/amazon_reviews_clean"

df_filtered.write.mode("overwrite").parquet(output_path)
print("Saved to disk")

In [ ]:
df_filtered = spark.read.parquet("/home/ajaganathan/scratch/spark_tmp/amazon_reviews_clean")
print(f"After clean up: {df_filtered.count()}")